# Predictive Graph Analytics for Anti-Money Laundering (AML)
## Exploratory Data Analysis & Typology Walkthrough

This notebook demonstrates how to load the processed transaction data, construct the directed weighted graph, inspect graph diagnostics, run typology detection rules, and explore feature importances.

In [ ]:
import sys
from pathlib import Path

# Set project root
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from src.graph.builder import build_transaction_graph
from src.graph.statistics import compute_graph_statistics, print_graph_summary
from src.detection.baseline import run_rule_based_detection

### 1. Load Processed Transactions & Accounts

In [ ]:
txns_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "processed_transactions.parquet")
accs_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "processed_accounts.parquet")
scored_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "account_risk_scores.parquet")

print(f"Transactions loaded: {len(txns_df):,}")
print(f"Accounts loaded: {len(accs_df):,}")
print(f"Laundering prevalence: {txns_df['is_laundering'].mean():.2%}")
scored_df[['account_id', 'aml_risk_score', 'risk_tier', 'rule_count']].head()

### 2. Graph Construction & Network Diagnostics

In [ ]:
G, multi_G = build_transaction_graph(txns_df, accs_df)
stats = compute_graph_statistics(G)
print_graph_summary(stats)

### 3. Rule Detection & Comparison with Ground Truth

In [ ]:
rules_eval_df, rule_metrics = run_rule_based_detection(txns_df, G, accs_df)
print("Rule Baseline Precision:", f"{rule_metrics['precision']:.4f}")
print("Rule Baseline Recall:   ", f"{rule_metrics['recall']:.4f}")
print("Rule Baseline F1:       ", f"{rule_metrics['f1_score']:.4f}")

### 4. Feature Importance Visualization

In [ ]:
from PIL import Image
img_path = PROJECT_ROOT / "reports" / "feature_importance.png"
if img_path.exists():
    img = Image.open(img_path)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()